# Extending GeoAgent with your own tools

GeoAgent is a thin coordinator over deepagents subagents. Bringing your own geospatial tooling is a matter of decorating a Python function with `@geo_tool` (or stamping metadata on an existing LangChain tool with `stamp_geo_meta`) and passing it to `GeoAgent(tools=[...])`.


## `@geo_tool` arguments

- `category` — one of `"map"`, `"qgis"`, `"data"`, `"ai"`, `"io"`. Used by the registry to filter and by the safety bridge to label confirm requests.
- `name` — optional override for the tool name (defaults to the function name).
- `description` — optional override (defaults to the function's docstring).
- `requires_confirmation` — when `True`, the factory wires this tool into deepagents' `interrupt_on` so it pauses for user approval (see [hitl_confirm.ipynb](hitl_confirm.ipynb)).
- `requires_packages` — packages whose absence makes the registry silently skip this tool (e.g. `("qgis",)`).
- `context_keys` — `GeoAgentContext` fields the tool reads at runtime. Informational; live runtime objects flow via closure, not via this list.


## Define a custom data tool

In [ ]:
from geoagent import GeoAgent, geo_tool


@geo_tool(
    category="data",
    description=(
        "Count features in a remote GeoJSON / Shapefile / GeoParquet URL."
        " Returns the feature count and the source CRS."
    ),
    requires_packages=["geopandas"],
)
def count_features_in_url(url: str) -> dict:
    """Open a vector dataset by URL with geopandas and count features.

    Args:
        url: A URL geopandas can open (GeoJSON, Shapefile, GeoParquet, ...).

    Returns:
        Dict with ``count`` (int) and ``crs`` (string).
    """
    import geopandas as gpd

    gdf = gpd.read_file(url)
    return {"count": int(len(gdf)), "crs": str(gdf.crs)}


count_features_in_url.metadata

## Use the tool from a chat

Pass the decorated tool via `tools=[...]`. The agent inspects the tool's description (the `@geo_tool` decorator promotes the docstring) to decide when to call it.


In [ ]:
agent = GeoAgent(tools=[count_features_in_url])
result = agent.chat(
    "How many features are in https://raw.githubusercontent.com/python-visualization/folium-example-data/main/us_states.json?"
)
print("success:", result.success)
print("executed:", result.executed_tools)
print("answer:", (result.answer_text or "")[:400])

## Retrofitting an existing LangChain tool with `stamp_geo_meta`

If you already have a tool built with the plain `@langchain_core.tools.tool` decorator (or imported from another package), you can enroll it in the GeoAgent registry without redecorating: `stamp_geo_meta` merges metadata into the tool in place.

In [ ]:
from langchain_core.tools import tool as lc_tool
from geoagent import stamp_geo_meta, get_geo_meta


@lc_tool
def list_supported_crs() -> list[str]:
    """Return a small set of common geographic / projected CRS strings."""
    return ["EPSG:4326", "EPSG:3857", "EPSG:32633"]


stamp_geo_meta(
    list_supported_crs,
    category="io",
    requires_confirmation=False,
    requires_packages=[],
)
get_geo_meta(list_supported_crs)

## Confirmation-required custom tools

Setting `requires_confirmation=True` flips the tool into the HITL path. The default `auto_approve_safe_only` callback rejects everything, so a fresh `GeoAgent(tools=[...])` will *plan* the call but cancel it. See [hitl_confirm.ipynb](hitl_confirm.ipynb) for the approval flow.

In [ ]:
@geo_tool(
    category="data",
    description="Pretend to clear a local raster cache.",
    requires_confirmation=True,
)
def clear_cache() -> str:
    """Pretend to clear the cache (no-op for the demo)."""
    return "cache cleared"


agent = GeoAgent(tools=[clear_cache])
result = agent.chat("Clear the local raster cache.")
print("executed:", result.executed_tools)
print("cancelled:", result.cancelled_tools)